# Notebook 03 — VaR and CVaR Estimator Implementation and Validation

**DS603 Group Project** · Tan Kok Yong (01426300)

This notebook:
1. Imports the estimator functions from `src/estimators.py` and validates them on synthetic data.
2. Computes VaR and CVaR at 95% and 99% confidence levels for each ticker and sub-period.
3. Presents a summary comparison table.



In [ ]:
# Install required packages (run once per Colab session)
!pip install -q wrds pandas numpy scipy statsmodels matplotlib seaborn pyarrow

import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import json

from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/DS603_Project'
sys.path.insert(0, PROJECT_ROOT)

from src.estimators import (
    historical_var, historical_cvar,
    parametric_normal_var, parametric_normal_cvar,
    parametric_t_var, parametric_t_cvar,
    fit_student_t, true_t_var, true_t_cvar,
    compute_all_estimators,
)

PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
FIG_DIR       = os.path.join(PROJECT_ROOT, 'outputs', 'figures')
TAB_DIR       = os.path.join(PROJECT_ROOT, 'outputs', 'tables')
MDL_DIR       = os.path.join(PROJECT_ROOT, 'models')
sns.set_theme(style='whitegrid', context='notebook')
TICKERS = ['JPM', 'AAPL', 'TSLA']
ALPHAS  = (0.95, 0.99)

## 1. Sanity check on synthetic data

We generate data from a known Normal(0, 1) distribution and verify that all estimators recover the true VaR and CVaR analytically.  
For N(0,1): VaR_{0.99} = Φ⁻¹(0.99) ≈ 2.326, CVaR_{0.99} = φ(Φ⁻¹(0.99)) / (1−0.99) ≈ 2.665.

In [ ]:
rng = np.random.default_rng(42)
N_SANITY = 100_000
synth = rng.standard_normal(N_SANITY)  # losses ~ N(0,1)

alpha = 0.99
true_var_norm  = stats.norm.ppf(alpha)
true_cvar_norm = stats.norm.pdf(stats.norm.ppf(alpha)) / (1 - alpha)

print(f'True VaR_{{0.99}}  (Normal): {true_var_norm:.6f}')
print(f'True CVaR_{{0.99}} (Normal): {true_cvar_norm:.6f}')
print()
print(f'Historical VaR:            {historical_var(synth, alpha):.6f}')
print(f'Historical CVaR:           {historical_cvar(synth, alpha):.6f}')
print(f'Parametric Normal VaR:     {parametric_normal_var(synth, alpha):.6f}')
print(f'Parametric Normal CVaR:    {parametric_normal_cvar(synth, alpha):.6f}')

# Student-t fit on Normal data should give high df (approaching Normal)
df_t, loc, scale = fit_student_t(synth)
print(f'\nStudent-t MLE on Normal data: df={df_t:.1f}, loc={loc:.4f}, scale={scale:.4f}')
print(f'Parametric Student-t VaR:  {parametric_t_var(synth, alpha, (df_t, loc, scale)):.6f}')
print(f'Parametric Student-t CVaR: {parametric_t_cvar(synth, alpha, (df_t, loc, scale)):.6f}')

**Expected result:** All estimators should recover values close to 2.326 (VaR) and 2.665 (CVaR) for N=100,000 draws. Small discrepancies are sampling error; they should vanish as N → ∞. Student-t degrees of freedom should be large (≫30), confirming it converges to Normal on Normal data.

## 2. Estimators on real data

In [ ]:
df = pd.read_parquet(os.path.join(PROCESSED_DIR, 'returns_clean.parquet'))
with open(os.path.join(MDL_DIR, 't_params.json')) as f:
    t_params_all = json.load(f)
df.head(3)

In [ ]:
records = []
for ticker in TICKERS:
    for period in ['full', 'calm', 'stressed']:
        if period == 'full':
            losses = df[df['ticker'] == ticker]['loss'].dropna().values
        else:
            losses = df[(df['ticker'] == ticker) & (df['period'] == period)]['loss'].dropna().values
        if len(losses) < 30:
            continue
        key = f'{ticker}_{period}'
        p = t_params_all.get(key)
        t_params = (p['df'], p['loc'], p['scale']) if p else None

        for alpha in ALPHAS:
            a_str = f'{int(alpha*100)}%'
            records.append({
                'Ticker': ticker, 'Period': period, 'Alpha': a_str, 'N': len(losses),
                'Hist VaR':   round(historical_var(losses, alpha) * 100, 4),
                'Hist CVaR':  round(historical_cvar(losses, alpha) * 100, 4),
                'Norm VaR':   round(parametric_normal_var(losses, alpha) * 100, 4),
                'Norm CVaR':  round(parametric_normal_cvar(losses, alpha) * 100, 4),
                't VaR':      round(parametric_t_var(losses, alpha, t_params) * 100, 4),
                't CVaR':     round(parametric_t_cvar(losses, alpha, t_params) * 100, 4),
            })

est_table = pd.DataFrame(records)
est_table.to_csv(os.path.join(TAB_DIR, '03_estimator_results.csv'), index=False)
est_table

## 3. Visualisation: VaR comparison by method, ticker, period

In [ ]:
# Focus on 99% VaR for the bar chart
sub99 = est_table[est_table['Alpha'] == '99%'].copy()

fig, axes = plt.subplots(1, len(TICKERS), figsize=(15, 5), sharey=False)
methods = ['Hist VaR', 'Norm VaR', 't VaR']
colors  = ['steelblue', 'crimson', 'seagreen']
x = np.arange(3)  # calm, stressed, full
periods = ['calm', 'stressed', 'full']
width = 0.22

for ax, ticker in zip(axes, TICKERS):
    sub = sub99[sub99['Ticker'] == ticker]
    for i, (method, color) in enumerate(zip(methods, colors)):
        vals = [sub[sub['Period'] == p][method].values[0] if len(sub[sub['Period'] == p]) > 0 else 0
                for p in periods]
        ax.bar(x + i * width - width, vals, width, label=method, color=color, alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(periods)
    ax.set_title(ticker)
    ax.set_ylabel('99% VaR (% daily loss)')
    ax.legend(fontsize=8)

plt.suptitle('99% VaR by Method, Ticker, and Sub-period', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '03_var_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:**  
- The **Normal estimator** consistently produces lower VaR values than historical or Student-t, especially in stressed periods — underestimating tail risk.  
- The **Student-t estimator** generally gives higher VaR, correctly accounting for heavy tails.  
- **Stressed** VaR estimates are substantially higher than calm estimates, validating that the sub-period split captures meaningful regime differences.  
- **TSLA** shows the highest VaR across all methods, consistent with its high-volatility profile.

## 4. VaR and CVaR on the loss distribution plot

In [ ]:
fig, axes = plt.subplots(1, len(TICKERS), figsize=(15, 5))
alpha = 0.99

for ax, ticker in zip(axes, TICKERS):
    losses = df[df['ticker'] == ticker]['loss'].dropna().values
    p = t_params_all[f'{ticker}_full']
    t_params = (p['df'], p['loc'], p['scale'])

    ax.hist(losses, bins=80, density=True, color='steelblue', alpha=0.5, label='Empirical losses')

    var_h = historical_var(losses, alpha)
    var_n = parametric_normal_var(losses, alpha)
    var_t = parametric_t_var(losses, alpha, t_params)
    cvar_h = historical_cvar(losses, alpha)
    cvar_n = parametric_normal_cvar(losses, alpha)
    cvar_t = parametric_t_cvar(losses, alpha, t_params)

    ymax = ax.get_ylim()[1]
    for val, label, color, ls in [
        (var_h,  'VaR Hist',  'steelblue', '-'),
        (var_n,  'VaR Norm',  'crimson',   '--'),
        (var_t,  'VaR t',     'seagreen',  '-'),
        (cvar_h, 'CVaR Hist', 'steelblue', ':'),
        (cvar_n, 'CVaR Norm', 'crimson',   '-.'),
        (cvar_t, 'CVaR t',    'seagreen',  ':'),
    ]:
        ax.axvline(val, color=color, ls=ls, lw=1.5, label=f'{label}={val*100:.2f}%')

    ax.set_xlim(-0.05, np.percentile(losses, 99.5))
    ax.set_title(ticker)
    ax.set_xlabel('Daily loss L = −RET')
    ax.legend(fontsize=6, loc='upper right')

plt.suptitle('99% VaR and CVaR on the loss distribution', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '03_var_cvar_on_dist.png'), dpi=150, bbox_inches='tight')
plt.show()